# C3 Ensemble clean Kaggle pipeline

This notebook is a thin, auditable launcher for the modular implementation in `src/c3_clean`. It never modifies legacy notebooks or raw CSV files. P0 must pass before P1-P3 can execute. All outputs are written under `/kaggle/working/c3_clean_artifacts/`.

In [ ]:
from pathlib import Path
import os, sys, subprocess, yaml, shutil

REPOSITORY_URL = 'https://github.com/FlynnBui399/vietnamese-emoji-emotion-recognition.git'
REPOSITORY_REF = 'main'
CLONE_IF_MISSING = True
CLONE_DESTINATION = Path('/kaggle/working/vietnamese-emoji-emotion-recognition')

def find_repository_root():
    direct = [Path.cwd(), Path('/kaggle/working')]
    for root in direct:
        if (root / 'configs' / 'c3_clean.yaml').is_file():
            return root.resolve()
    for search_root in (Path('/kaggle/input'), Path('/kaggle/working')):
        if search_root.is_dir():
            for match in search_root.rglob('configs/c3_clean.yaml'):
                return match.parent.parent.resolve()
    return None

REPO_ROOT = find_repository_root()
if REPO_ROOT is None and CLONE_IF_MISSING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Automatic cloning is intended for Kaggle')
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
        REPOSITORY_URL, str(CLONE_DESTINATION),
    ])
    REPO_ROOT = CLONE_DESTINATION.resolve()
if REPO_ROOT is None:
    raise FileNotFoundError('Attach the updated repository or enable CLONE_IF_MISSING')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)

In [ ]:
# Runtime values are explicit and are recorded in every resolved seed config.
INSTALL_DEPENDENCIES = False
DATA_DIR_OVERRIDE = None       # Example: '/kaggle/input/c3-clean-inputs/data/vigoemotions'
EMOJI2VEC_OVERRIDE = None      # Example: '/kaggle/input/c3-clean-inputs/data/emoji2vec.bin'
VISOBERT_MODEL_OVERRIDE = None # Set to an attached HF snapshot when Kaggle internet is off.
RESUME_ARTIFACTS_SOURCE = None # Example: '/kaggle/input/c3-clean-resume/c3_clean_artifacts'
NUM_GPUS = 1          # Set exactly 2 only in a Kaggle two-GPU session.
PRECISION = 'fp32'    # One of: fp32, fp16, bf16. No automatic fallback.
BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 1
MAX_LENGTH = 128

RUN_EXPERIMENTS = [
    'A0_controlled_text_BCE',
    'A1_controlled_text_ASL',
    'A2_controlled_ASL_Emoji',
    'A3_controlled_ASL_Emoji_CB',
]
OPTIONAL_EXPERIMENTS = [
    'Emoji-random-control',
    'Emoji-shuffle-control',
    'Emoji-zero-control',
    'C3-RDrop',
    'C3-extended-matched',
]

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')])

working_artifacts = Path('/kaggle/working/c3_clean_artifacts')
if RESUME_ARTIFACTS_SOURCE is not None:
    resume_source = Path(RESUME_ARTIFACTS_SOURCE)
    if not resume_source.is_dir():
        raise FileNotFoundError(f'Resume artifact directory not found: {resume_source}')
    shutil.copytree(resume_source, working_artifacts, dirs_exist_ok=True)
    print('Restored prior artifacts from:', resume_source)

base_config_path = REPO_ROOT / 'configs' / 'c3_clean.yaml'
config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
config['paths']['repository_root'] = str(REPO_ROOT)
if DATA_DIR_OVERRIDE is not None:
    config['paths']['data_dir_override'] = DATA_DIR_OVERRIDE
if EMOJI2VEC_OVERRIDE is not None:
    config['paths']['emoji2vec'] = EMOJI2VEC_OVERRIDE
if VISOBERT_MODEL_OVERRIDE is not None:
    config['model']['model_name'] = VISOBERT_MODEL_OVERRIDE
config['runtime']['num_gpus'] = NUM_GPUS
config['runtime']['precision'] = PRECISION
config['training']['batch_size'] = BATCH_SIZE
config['training']['gradient_accumulation'] = GRADIENT_ACCUMULATION
config['model']['max_length'] = MAX_LENGTH
runtime_config_path = Path('/kaggle/working/c3_clean_runtime.yaml')
runtime_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Runtime config:', runtime_config_path)

## P0 - source audit and artifact-only reconstruction

This cell hashes and audits the raw splits, inventories attached legacy bundles, verifies available checkpoints, averages the three validation/test probability arrays, and fits new thresholds only on averaged validation probabilities. A data discrepancy exits with status 2 after writing the discrepancy report.

In [ ]:
from src.c3_clean.run_experiments import main
P0_STATUS = main(['--config', str(runtime_config_path), '--priority', 'P0'])
if P0_STATUS != 0:
    raise RuntimeError('P0 stopped. Inspect /kaggle/working/c3_clean_artifacts/data_discrepancies.md before continuing.')

## P1 - canonical C3 seeds and ensemble

Run this cell across sessions as needed. Completed seeds are skipped; interrupted seeds resume from `last_checkpoint.pt`.

In [ ]:
P1_STATUS = main([
    '--config', str(runtime_config_path),
    '--priority', 'P1',
    '--run-experiments', 'A3_controlled_ASL_Emoji_CB',
])
assert P1_STATUS == 0

## P2 - controlled A0-A3 ablation and matched ensembles

In [ ]:
P2_STATUS = main([
    '--config', str(runtime_config_path),
    '--priority', 'P2',
    '--run-experiments', *RUN_EXPERIMENTS,
])
assert P2_STATUS == 0

## P3 - optional diagnostics

Run only after P0-P2 are complete. These results remain separate from the canonical C3 headline result.

In [ ]:
P3_STATUS = main([
    '--config', str(runtime_config_path),
    '--priority', 'P3',
    '--run-experiments', *OPTIONAL_EXPERIMENTS,
])
assert P3_STATUS == 0

## Final package

Create the required ZIP only after the selected priorities have completed.

In [ ]:
from src.c3_clean.run_experiments import package_kaggle_artifacts
archive = package_kaggle_artifacts(Path('/kaggle/working/c3_clean_artifacts'))
print(archive)